In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('../data/raw/ecommerce.csv', encoding='ISO-8859-1')

In [4]:
df.head()

,Order_ID,Customer_ID,Date,Age,Gender,City,Product_Category,Unit_Price,Quantity,Discount_Amount,Total_Amount,Payment_Method,Device_Type,Session_Duration_Minutes,Pages_Viewed,Is_Returning_Customer,Delivery_Time_Days,Customer_Rating
0,ORD_001337,CUST_01337,2023-01-01,27,Female,Bursa,Toys,54.28,1,0.00,54.28,Debit Card,Mobile,4,14,True,8,5
1,ORD_004885,CUST_04885,2023-01-01,42,Male,Konya,Toys,244.90,1,0.00,244.90,Credit Card,Mobile,11,3,True,3,3
2,ORD_004507,CUST_04507,2023-01-01,43,Female,Ankara,Food,48.15,5,0.00,240.75,Credit Card,Mobile,7,8,True,5,2
3,ORD_000645,CUST_00645,2023-01-01,32,Male,Istanbul,Electronics,804.06,1,229.28,574.78,Credit Card,Mobile,8,10,False,1,4
4,ORD_000690,CUST_00690,2023-01-01,40,Female,Istanbul,Sports,755.61,5,0.00,3778.05,Cash on Delivery,Desktop,21,10,True,7,4


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Order_ID                  5000 non-null   object 
 1   Customer_ID               5000 non-null   object 
 2   Date                      5000 non-null   object 
 3   Age                       5000 non-null   int64  
 4   Gender                    5000 non-null   object 
 5   City                      5000 non-null   object 
 6   Product_Category          5000 non-null   object 
 7   Unit_Price                5000 non-null   float64
 8   Quantity                  5000 non-null   int64  
 9   Discount_Amount           5000 non-null   float64
 10  Total_Amount              5000 non-null   float64
 11  Payment_Method            5000 non-null   object 
 12  Device_Type               5000 non-null   object 
 13  Session_Duration_Minutes  5000 non-null   int64  
 14  Pages_Vi

In [6]:
df['Date'] = pd.to_datetime(df['Date'])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Order_ID                  5000 non-null   object        
 1   Customer_ID               5000 non-null   object        
 2   Date                      5000 non-null   datetime64[ns]
 3   Age                       5000 non-null   int64         
 4   Gender                    5000 non-null   object        
 5   City                      5000 non-null   object        
 6   Product_Category          5000 non-null   object        
 7   Unit_Price                5000 non-null   float64       
 8   Quantity                  5000 non-null   int64         
 9   Discount_Amount           5000 non-null   float64       
 10  Total_Amount              5000 non-null   float64       
 11  Payment_Method            5000 non-null   object        
 12  Device_Type         

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
(df['Quantity'] <= 0).sum()
(df['Unit_Price'] <= 0).sum()
(df['Total_Amount'] <= 0).sum()

np.int64(0)

In [9]:
df['Check_Total'] = df['Unit_Price'] * df['Quantity'] - df['Discount_Amount']
(df['Total_Amount'] != df['Check_Total']).sum()

np.int64(706)

In [10]:
df['Total_Amount'] = df['Check_Total']
df.drop(columns=['Check_Total'], inplace=True)

In [11]:
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['Weekday'] = df['Date'].dt.day_name()

In [12]:
customer_revenue = df.groupby('Customer_ID')['Total_Amount'].sum()

In [13]:
print("Total Revenue:", df['Total_Amount'].sum())

df.groupby('City')['Total_Amount'].sum().sort_values(ascending=False).head()

df.groupby('Product_Category')['Total_Amount'].sum().sort_values(ascending=False)

df.groupby('Is_Returning_Customer')['Total_Amount'].sum()

Total Revenue: 4915544.57


Is_Returning_Customer
False    1967523.73
True     2948020.84
Name: Total_Amount, dtype: float64

In [14]:
df.to_csv('../data/cleaned/cleaned_data.csv', index=False)

Final cleaned dataset ready for export

RFM Analysis starts here 

In [15]:
import datetime as dt

snapshot_date = df['Date'].max() + dt.timedelta(days=1)

In [16]:
rfm = df.groupby('Customer_ID').agg({
    'Date': lambda x: (snapshot_date - x.max()).days,
    'Order_ID': 'count',
    'Total_Amount': 'sum'
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']

rfm.head()

,Recency,Frequency,Monetary
Customer_ID,,,
CUST_00001,124,1,219.32
CUST_00002,394,1,332.34
CUST_00003,439,1,66.44
CUST_00004,72,1,112.65
CUST_00005,311,1,349.45


In [18]:
rfm['R_score'] = pd.qcut(rfm['Recency'], 4, labels=[4,3,2,1])

rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1,2,3,4])

rfm['M_score'] = pd.qcut(rfm['Monetary'], 4, labels=[1,2,3,4])

In [19]:
rfm['RFM_Score'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)

In [20]:
def segment_customer(row):
    if row['RFM_Score'] == '444':
        return 'VIP'
    elif row['F_score'] == 4:
        return 'Loyal'
    elif row['R_score'] == 1:
        return 'At Risk'
    else:
        return 'Regular'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)

In [21]:
rfm['Segment'].value_counts()

Segment
Regular    2821
Loyal      1162
At Risk     929
VIP          88
Name: count, dtype: int64

In [22]:
rfm.groupby('Segment')['Monetary'].sum().sort_values(ascending=False)

Segment
Regular    2794424.08
At Risk     923514.87
Loyal       905359.91
VIP         292245.71
Name: Monetary, dtype: float64

In [23]:
df = df.merge(rfm[['Segment']], on='Customer_ID', how='left')

In [24]:
rfm.to_csv('../data/cleaned/rfm_data.csv')
df.to_csv('../data/cleaned/final_data_with_segments.csv', index=False)